In [9]:
import sys

!{sys.executable} -m pip install \
langchain-community \
langchain-text-splitters \
langchain-groq \
langchain-huggingface \
sentence-transformers \
pypdf \
scikit-learn

from langchain_community.document_loaders import PyPDFLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_huggingface import HuggingFaceEmbeddings

from sklearn.metrics.pairwise import cosine_similarity

from langchain_groq import ChatGroq

import numpy as np


pdf_path = "/Users/Aryaanil/Documents/survey cdc.pdf"


loader = PyPDFLoader(pdf_path)

documents = loader.load()


text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)


texts = [doc.page_content for doc in chunks]


embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


document_embeddings = embedding_model.embed_documents(texts)

document_embeddings = np.array(document_embeddings)


groq_api_key = "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"


llm = ChatGroq(
    groq_api_key=groq_api_key,
    model_name="llama-3.3-70b-versatile"
)


questions = [
    "What is the main topic of the PDF?",
    "Explain one important concept from the notes.",
    "What tools or libraries are mentioned?",
    "Summarize the document in simple words.",
    "What are the key learning points?"
]


def retrieve_documents(query, k=3):

    query_embedding = embedding_model.embed_query(query)

    similarity_scores = cosine_similarity(
        [query_embedding],
        document_embeddings
    )[0]

    top_indices = similarity_scores.argsort()[::-1][:k]

    return [texts[i] for i in top_indices]


for question in questions:

    retrieved_docs = retrieve_documents(question)

    context = "\n".join(retrieved_docs)

    prompt = f"""
    Answer the question using the context below.

    Context:
    {context}

    Question:
    {question}
    """

    response = llm.invoke(prompt)

    print("\nQUESTION:")
    print(question)

    print("\nANSWER:")
    print(response.content)

    print("\n" + "=" * 60)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


QUESTION:
What is the main topic of the PDF?

ANSWER:
The main topic of the survey appears to be related to academic and student-related matters, specifically focusing on the student community, with an emphasis on stress and how students handle difficult situations.


QUESTION:
Explain one important concept from the notes.

ANSWER:
One important concept from the notes is the idea of assessing and measuring happiness in daily life. This is reflected in question 5 of Section B, which asks respondents to rate their happiness on a scale of 1-10. This concept is significant because it acknowledges that happiness is a subjective and personal experience that can vary from person to person, and that understanding an individual's level of happiness can provide insight into their overall emotional well-being.


QUESTION:
What tools or libraries are mentioned?

ANSWER:
None, the context provided does not mention any specific tools or libraries. It appears to be a survey questionnaire focused on 